In [3]:
from pathlib import Path
import math
import re
from collections import defaultdict
import pandas as pd
from IPython.display import display, Markdown

# --- Config ---
DATA_DIR = Path("../data")
CURRENT_GRID = {
    500: [19, 38, 64, 96, 128],
    1000: [19, 38, 64, 96, 128],
    1500: [19, 38, 64, 96, 128],
    2000: [96, 128],
    2500: [96, 128],
    3000: [96, 128],
}
EXPECTED_STATES = {
    0: ["demag"],
    1: ["healthy", "misalignment", "front_ball", "rear_ball"],
    2: ["healthy", "misalignment", "front_ball", "rear_ball"],
    3: ["healthy", "misalignment", "front_ball", "rear_ball"],
}
EXPECTED_STATE_RANK = {
    motor: {state: i for i, state in enumerate(states)}
    for motor, states in EXPECTED_STATES.items()
}
NOMINAL_SAMPLING_RATE_HZ = 1000.0
MEAN_RATE_TOLERANCE_HZ = 10.0

# File-name pattern:
# analize_<state><id>[_quality]_<speed>rpm_<load-current>mA_<power-source>.csv
FILE_RE = re.compile(
    r"^analize_(?P<state>[a-z_]+?)(?P<id>\d+)(?:_(?P<quality>ENV|SF))?_(?P<rpm>\d+)rpm_(?P<ma>\d+)mA_(?P<source>[a-zA-Z0-9]+)\.csv$",
    re.IGNORECASE,
)


def decode_motor_from_id(exp_id: str) -> int:
    motor_prefix = exp_id[:-1]
    return int(motor_prefix or "0")


def currents_for_rpm(rpm: int) -> list[int]:
    return CURRENT_GRID.get(rpm, [])


def load_dt_seconds(csv_path: Path) -> pd.Series:
    recording = pd.read_csv(csv_path, usecols=["dt"])
    dt_s = pd.to_numeric(recording["dt"], errors="coerce").dropna()
    dt_s = dt_s[dt_s > 0]
    if not dt_s.empty:
        return dt_s

    timestamps_us = pd.to_numeric(
        pd.read_csv(csv_path, usecols=["t_us"])["t_us"],
        errors="coerce",
    ).dropna()
    if timestamps_us.size < 2:
        return pd.Series(dtype=float)

    dt_s = timestamps_us.diff().dropna() * 1e-6
    return dt_s[dt_s > 0]


sampling_stats = defaultdict(lambda: {"dt_sum_s": 0.0, "dt_sq_sum_s2": 0.0, "dt_count": 0})
read_errors = []

all_csv = sorted(DATA_DIR.glob("analize_*.csv"))

for f in all_csv:
    if f.name.lower() == "analize_0rpm_0ma.csv":
        continue

    name_upper = f.name.upper()
    if "_ENV_" in name_upper or "_SF_" in name_upper:
        continue

    m = FILE_RE.match(f.name)
    if not m:
        continue

    if m.group("source").lower() == "bat":
        continue

    state = m.group("state").lower()
    exp_id = m.group("id")
    rpm = int(m.group("rpm"))
    ma = int(m.group("ma"))
    motor = decode_motor_from_id(exp_id)

    if motor not in EXPECTED_STATES or state not in EXPECTED_STATES[motor]:
        continue
    if rpm not in CURRENT_GRID or ma not in currents_for_rpm(rpm):
        continue

    try:
        dt_s = load_dt_seconds(f)
        if dt_s.empty:
            continue

        key = (motor, state, rpm, ma)
        stats = sampling_stats[key]
        stats["dt_sum_s"] += float(dt_s.sum())
        stats["dt_sq_sum_s2"] += float((dt_s**2).sum())
        stats["dt_count"] += int(dt_s.size)
    except Exception as e:
        read_errors.append((f.name, str(e)))

rows = []
for motor in sorted(EXPECTED_STATES):
    for state in EXPECTED_STATES[motor]:
        for rpm in CURRENT_GRID:
            for ma in currents_for_rpm(rpm):
                key = (motor, state, rpm, ma)
                stats = sampling_stats.get(key)

                if stats and stats["dt_count"] > 0:
                    mean_dt_s = stats["dt_sum_s"] / stats["dt_count"]
                    mean_dt_us = mean_dt_s * 1e6
                    mean_rate_hz = 1.0 / mean_dt_s
                    duration_min = stats["dt_sum_s"] / 60.0
                    variance_s2 = (stats["dt_sq_sum_s2"] / stats["dt_count"]) - (mean_dt_s**2)
                    variance_s2 = max(variance_s2, 0.0)
                    std_dt_us = math.sqrt(variance_s2) * 1e6
                    rate_out_of_tolerance = abs(mean_rate_hz - NOMINAL_SAMPLING_RATE_HZ) > MEAN_RATE_TOLERANCE_HZ
                    is_missing = False
                else:
                    mean_dt_us = None
                    mean_rate_hz = None
                    duration_min = None
                    std_dt_us = None
                    rate_out_of_tolerance = False
                    is_missing = True

                rows.append(
                    {
                        "motor": motor,
                        "state": state,
                        "rpm": rpm,
                        "mA": ma,
                        "sample_intervals": "-" if is_missing else stats["dt_count"],
                        "duration_min": "-" if is_missing else f"{duration_min:.3f}",
                        "mean_dt_us": "-" if is_missing else f"{mean_dt_us:.3f}",
                        "sampling_rate_hz": "-" if is_missing else f"{mean_rate_hz:.3f}",
                        "std_dt_us": "-" if is_missing else f"{std_dt_us:.3f}",
                        "rate_out_of_tolerance": rate_out_of_tolerance,
                        "is_missing": is_missing,
                    }
                )

sampling_df = pd.DataFrame(rows)
sampling_df["state_rank"] = sampling_df.apply(
    lambda r: EXPECTED_STATE_RANK.get(r["motor"], {}).get(r["state"], 999),
    axis=1,
)
sampling_df = sampling_df.sort_values(["motor", "state_rank", "rpm", "mA"]).reset_index(drop=True)

view_cols = [
    "motor",
    "state",
    "rpm",
    "mA",
    "sample_intervals",
    "duration_min",
    "mean_dt_us",
    "sampling_rate_hz",
    "std_dt_us",
    "rate_out_of_tolerance",
]


def style_table(display_df):
    style_df = pd.DataFrame("", index=display_df.index, columns=display_df.columns)
    style_df.loc[display_df["is_missing"], "sampling_rate_hz"] = "color: red; font-weight: 700;"
    style_df.loc[display_df["rate_out_of_tolerance"].eq(True), :] = "color: red; font-weight: 700;"
    return style_df


display(Markdown("## Laboratory power supply only"))
for motor in sorted(EXPECTED_STATES):
    motor_df = sampling_df[sampling_df["motor"] == motor].copy()
    print(f"\nMotor {motor}")
    display_df = motor_df[view_cols + ["is_missing"]].copy()
    display_df["rate_out_of_tolerance"] = display_df["rate_out_of_tolerance"].astype("object")
    display_df.loc[display_df["is_missing"], "rate_out_of_tolerance"] = "-"
    styled = (
        display_df.style.apply(style_table, axis=None)
        .hide(axis="columns", subset=["is_missing"])
        .hide(axis="index")
    )
    display(styled)

missing_count = int(sampling_df["is_missing"].sum())
rate_violation_count = int(sampling_df["rate_out_of_tolerance"].sum())
print(f"\nMissing combinations: {missing_count}")
print(f"Sampling-rate tolerance violations found: {rate_violation_count}")

if read_errors:
    print("\nSome files could not be read:")
    for fname, err in read_errors[:10]:
        print(f"- {fname}: {err}")
    if len(read_errors) > 10:
        print(f"... and {len(read_errors) - 10} more files with errors")

## Laboratory power supply only


Motor 0


motor,state,rpm,mA,sample_intervals,duration_min,mean_dt_us,sampling_rate_hz,std_dt_us,rate_out_of_tolerance
0,demag,500,19,-,-,-,-,-,-
0,demag,500,38,-,-,-,-,-,-
0,demag,500,64,-,-,-,-,-,-
0,demag,500,96,-,-,-,-,-,-
0,demag,500,128,-,-,-,-,-,-
0,demag,1000,19,-,-,-,-,-,-
0,demag,1000,38,-,-,-,-,-,-
0,demag,1000,64,-,-,-,-,-,-
0,demag,1000,96,-,-,-,-,-,-
0,demag,1000,128,-,-,-,-,-,-



Motor 1


motor,state,rpm,mA,sample_intervals,duration_min,mean_dt_us,sampling_rate_hz,std_dt_us,rate_out_of_tolerance
1,healthy,500,19,-,-,-,-,-,-
1,healthy,500,38,-,-,-,-,-,-
1,healthy,500,64,-,-,-,-,-,-
1,healthy,500,96,335111,5.585,1000.000,1000.000,8.542,False
1,healthy,500,128,-,-,-,-,-,-
1,healthy,1000,19,-,-,-,-,-,-
1,healthy,1000,38,-,-,-,-,-,-
1,healthy,1000,64,548611,9.144,1000.000,1000.000,8.092,False
1,healthy,1000,96,263156,4.386,1000.000,1000.000,7.515,False
1,healthy,1000,128,-,-,-,-,-,-



Motor 2


motor,state,rpm,mA,sample_intervals,duration_min,mean_dt_us,sampling_rate_hz,std_dt_us,rate_out_of_tolerance
2,healthy,500,19,262248,4.371,1000.000,1000.000,1.720,False
2,healthy,500,38,263559,4.393,1000.000,1000.000,1.655,False
2,healthy,500,64,262119,4.369,1000.000,1000.000,1.976,False
2,healthy,500,96,269344,4.489,1000.000,1000.000,1.994,False
2,healthy,500,128,278412,4.640,1000.000,1000.000,1.711,False
2,healthy,1000,19,262528,4.375,1000.000,1000.000,2.024,False
2,healthy,1000,38,265186,4.420,1000.000,1000.000,2.083,False
2,healthy,1000,64,267706,4.462,1000.000,1000.000,2.094,False
2,healthy,1000,96,262069,4.368,1000.000,1000.000,1.991,False
2,healthy,1000,128,265191,4.420,1000.000,1000.000,1.802,False



Motor 3


motor,state,rpm,mA,sample_intervals,duration_min,mean_dt_us,sampling_rate_hz,std_dt_us,rate_out_of_tolerance
3,healthy,500,19,222798,3.713,1000.000,1000.000,2.091,False
3,healthy,500,38,266147,4.436,1000.000,1000.000,2.012,False
3,healthy,500,64,299968,4.999,1000.000,1000.000,1.922,False
3,healthy,500,96,264289,4.405,1000.000,1000.000,2.062,False
3,healthy,500,128,268417,4.474,1000.000,1000.000,1.728,False
3,healthy,1000,19,262516,4.375,1000.000,1000.000,1.885,False
3,healthy,1000,38,262481,4.375,1000.000,1000.000,1.875,False
3,healthy,1000,64,262457,4.374,1000.000,1000.000,2.038,False
3,healthy,1000,96,262341,4.372,1000.000,1000.000,1.810,False
3,healthy,1000,128,262653,4.378,1000.000,1000.000,1.767,False



Missing combinations: 55
Sampling-rate tolerance violations found: 0
